<a href="https://colab.research.google.com/github/adhiss387-code/invoice-parser-agent/blob/main/02_Fraud_Detection_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report
from google.colab import files

# ==========================================
# 1. LOAD FILE
# ==========================================
file_name = "transaction_fraud_test_dataset_1030_2.xlsx"
if not os.path.exists(file_name):
    print(f"⚠️ '{file_name}' not found! Uploading now...")
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)
print(f"✅ Loaded '{file_name}' with {len(df)} transactions.")

# ==========================================
# 2. TRAIN ISOLATION FOREST & DETECT FRAUD
# ==========================================
features = ['amount', 'hour', 'vendor_risk_score', 'account_age_days',
            'location_match', 'transactions_last_24h']

# Sanity check: make sure all expected columns exist before proceeding
missing_cols = [c for c in features if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns in file: {missing_cols}. "
                      f"Available columns: {df.columns.tolist()}")

X = df[features].copy()

# Encode location_match if it's categorical (e.g. Yes/No, True/False)
if X['location_match'].dtype == 'object':
    X['location_match'] = X['location_match'].map(
        {'Yes': 1, 'No': 0, 'True': 1, 'False': 0, True: 1, False: 0}
    )

# Fixed contamination rate — do NOT tie this to a known fraud count,
# since real monthly/weekly data won't have one. Tune this % based on
# your organization's historical fraud rate.
contamination_rate = 0.03

model = IsolationForest(contamination=contamination_rate, random_state=42, n_estimators=200)
model.fit(X)

# Predict & Score
df['AI_Flagged_Fraud'] = np.where(model.predict(X) == -1, 1, 0)
df['Anomaly_Score'] = model.decision_function(X)
flagged_df = df[df['AI_Flagged_Fraud'] == 1].sort_values(by='Anomaly_Score')

# ==========================================
# 3. DISPLAY RESULTS & DOWNLOAD AUDIT REPORT
# ==========================================
print(f"\n🚨 AI Model Flagged {len(flagged_df)} Suspicious Transactions "
      f"out of {len(df)} ({len(flagged_df)/len(df)*100:.1f}%).")

# Only runs if you have ground-truth labels (e.g. testing on synthetic data)
if 'label' in df.columns:
    print("\n📊 Accuracy Report:")
    print(classification_report(df['label'], df['AI_Flagged_Fraud'],
                                 target_names=['Normal', 'Fraud']))

display_cols = [c for c in ['transaction_id', 'amount', 'hour', 'vendor_risk_score',
                             'location_match', 'Anomaly_Score'] if c in flagged_df.columns]
print("\nTop Flagged Audit Items:")
display(flagged_df[display_cols].head(10))

# Save & Auto-Download Excel Report
output_report = "FLAGGED_AUDIT_REPORT.xlsx"
with pd.ExcelWriter(output_report, engine='openpyxl') as writer:
    flagged_df.to_excel(writer, index=False, sheet_name='Suspicious Items')
print(f"\n📁 Exported audit ledger to '{output_report}'!")
files.download(output_report)

⚠️ 'transaction_fraud_test_dataset_1030_2.xlsx' not found! Uploading now...


Saving transaction_fraud_test_dataset_1030-1.xlsx to transaction_fraud_test_dataset_1030-1.xlsx
✅ Loaded 'transaction_fraud_test_dataset_1030-1.xlsx' with 1030 transactions.

🚨 AI Model Flagged 31 Suspicious Transactions out of 1030 (3.0%).

📊 Accuracy Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      1000
       Fraud       0.97      1.00      0.98        30

    accuracy                           1.00      1030
   macro avg       0.98      1.00      0.99      1030
weighted avg       1.00      1.00      1.00      1030


Top Flagged Audit Items:


,transaction_id,amount,hour,vendor_risk_score,location_match,Anomaly_Score
596,TXN01009,111176.15,0,0.835,0,-0.138200
964,TXN01013,103711.16,3,0.996,0,-0.135371
382,TXN01028,71355.05,1,0.933,1,-0.132261
984,TXN01023,80921.15,4,0.886,1,-0.131277
683,TXN01012,102088.04,2,0.788,1,-0.128981
672,TXN01008,110855.85,1,0.810,0,-0.128981
686,TXN01011,112987.95,3,0.956,0,-0.124920
129,TXN01030,109997.49,2,0.960,0,-0.118618
178,TXN01016,45690.92,1,0.844,1,-0.118582
798,TXN01024,112701.72,1,0.928,0,-0.117614



📁 Exported audit ledger to 'FLAGGED_AUDIT_REPORT.xlsx'!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>